In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from HHO_kernel import HHO_cell

# We import a few functions to test the HHO kernel to avoid cluttering the notebook
from test_utils import plot_basis 
    # Plot the basis of the polynomial space for the cell unkowns
from test_utils import test_HHO_computation 
    # Illustrate the result of L2 projection / reconstruction problem / local cell problem
from test_utils import test_HHO_convergence
    # Make convergence curves for different computations on the cell.

plt.rcParams.update({
    # Make tick labels point inside
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    # Set thicker axis lines (spines)
    'axes.linewidth': 1.5,
})

# Define a prototypical cell

In this notebook, we validate the results of the computations on the cell. To illustrate such results, we define here a protoypical `HHO_cell` object named `pcell` that performs these computations on the (arbitrary) domain $T = (2,2.5)$ and uses polynomials of degree `k = 5` for the cell unknowns. `pcell` shall be used throughout the notebook. We choose to work with a monomial basis. Alternatively, a basis consisting of Legendre polynomials is also available. In this case, change `basis_type` to `"legendre"`.

In [ ]:
k = 5 # cell degree
xL = 2.0 # left boundary
xR = 2.5 # right boundary
basis_type = "monomial"
pcell = HHO_cell(xL,xR,k,basis_type)
xx = np.linspace(xL,xR,50) # grid that will be used for visualization

# Check the computation of the monomial basis functions

In [ ]:
plot_basis(pcell, xx)

# Validation of the implementation of the L2-projection

On the cell, we frequently need to compute $L^2$-orthogonal projection of some function on the space $\mathbb{P}_k(T)$ for some polynomial degree $k$. 
To test the computation of the associated mass matrix analytically, we first compute the mass matrix for the case of a low polynomial degree (k=2).

In [ ]:
# Define test example
cell_test_mass = HHO_cell(-1,1,2)
print("Computed mass matrix:")
print(np.round(cell_test_mass.mass_matrix,3))

This is exactly the expected result (up to computer precision).

The inversion of the mass matrix is done by a Cholesky decomposition (from `scipy.linalg`). We check here that we use the interface to the Cholesky decomposition correctly by recompputing the mass matrix from it.

In [ ]:
# Compute the actual mass matrix from the computed cholesky decomposition
mass = cell_test_mass.mass_cho
lower = cell_test_mass._mass_cho_lower
if lower:
    mass = np.tril(mass) @ np.tril(mass).T 
else:
    mass = np.triu(mass).T @ np.triu(mass)
print("Reconstructed mass matrix from Cholesky decomposition:")
print(np.round(mass,3))

Indeed, the result corresponds to the original mass matrix.

Next we show the computed projection of some polynomials and a cosine function on the cell and we compute the maximum error at the points of the grid `xx`.

In [ ]:
test_HHO_computation("L2 projection", pcell, xx)

As expected, the projection of a 5th order polynomial is exact on the prototypical cell, for which we use polynomial unknowns of degree 5. 
This is no longer true if we go beyond the polynomial order of the cell unknowns, but the approximations of the other two functions still look all right.

### Convergence

Denoting $\Pi_H^k$ the piecewise $L^2$ projection on a grid of mesh size $H$ for a fixed domain $\Omega$, the projection error of any sufficiently regular function $f$ should satisfy
$$
\lVert f - \Pi^k_H(f) \rVert_{L^2(\Omega)} \leq C h^{k+1},
$$
and
$$
\lVert f - \Pi^k_H(f) \rVert_{H^1(\Omega)} \leq C h^{k},
$$
where $h$ is the grid spacing and $k$ the polynomial degree and the $H^1$-norm understood in the broken sense. This property is validated below with $f(x) = \cos(4\pi x)$ and $\Omega = (0,1)$.

In [ ]:
# !!! Changing the basis type does not yet affect the basis used in the convergence computation
test_HHO_convergence("L2 projection", test_degrees=[1, 2, 5, 6])

The order of convergence matches the theoretical prediction.
For the highest-order case, the convergence rate starts to worsen for the smallest values of $h$ due to the accumulation of rounding errors.
We conclude that the $L^2$ projection on the cell is implemented correctly.

# Validation of the computation of the reconstruction for the potential

Let us recall the definition of the reconstruction for the potential used in the HHO method. 
Let $\hat{u} = (u_T, u_{\partial T}) \in \mathbb{P}_k(T) \times \mathbb{P}_k(T)^2$ denote cell and face unknowns on $T$ (in the 1D case, we have $\mathbb{P}_k(T)^2 = \mathbb{R}^2$).
The reconstruction $R_T \hat{u} \in \mathbb{P}_{k+1}(T)$ is the unique solution to 
$$
\left( \nabla R_T \hat{u}, \nabla q \right)_{L^2(T)} = - \left(u_T, \Delta q \right)_{L^2(T)} + \sum_{F \in \partial T} \left( u_F, \vec{n}_F \cdot \nabla q \right), \\
\int_T R_T \hat{u} = \int_T u_T,
$$
for all $q \in \mathbb{P}_{k+1}(T)$.

To solve this problem, we first need to compute the stiffness matrix on $\mathbb{P}_{k+1}(T)$, in which we neglect the constant basis function to obtain a well-posed problem on $\mathbb{P}_{k+1}(T)/\mathbb{R}$.
We check the value of this stiffness matrix on a small example.

In [ ]:
print("Computed stiffness matrix for the reconstruction problem:")
print(np.round(cell_test_mass.reconstruction_matrix,3))

This is exactly the expected stiffness matrix, as can be confirmed by analytical integrations.

After solving the linear system on $\mathbb{P}_{k+1}(T)/\mathbb{R}$, the average of $\int_T R_T \hat{u}$ is set through a post-processing step.

Given any function $u \in H^1(T)$, we can compute its HHO interpolate $\hat{\Pi}^k_T u$ (consisting of its $L^2$ projection on $\mathbb{P}_{k}(T)$ and the values at the 2 faces), from which we can then compute the reconstruction $R_T\hat{\Pi}^k_T u \in \mathbb{P}_{k+1}(T)$.
Let us show some examples of functions $u$ and the reconstruction obtained in this way. We also compute, for each function, the error of the reconstruction at the two vertices of $T$. It is known in the specific case of the 1D reconstruction that $R_T \hat{u}$ should equal $u_{\partial T}$ at the vertices.

In [ ]:
test_HHO_computation("Reconstruction", pcell, xx)

In contrast to the $L^2$-projection, the reconstruction is exact even for the 6th order polynomial with cell degree 5 on the prototypical cell.
This is exactly what we needed to find, since , with cell degree $k=5$, the reconstruction is the elliptic projection $\mathbb{P}_{6}(T)$.

We can furthermore confirm that, even for the cosine function, which does not allow for an exact representation by the reconstruction, the error at the faces vanishes up to rounding errors.

### Convergence

We consider here the case of a fixed domain $\Omega$ discretized by a grid of cells of decreasing size.
For any function $u \in H^1(\Omega)$, let $\hat{\Pi}^k_H u$ denote the interpolate of $u$ in the discrete HHO space associated to the whole grid.
The HHO theory proves the following error estimates for any sufficiently regular function $u$:
$$
\lVert u - R_H(\hat{\Pi}^k_H u) \rVert_{L^2(\Omega)} \leq C h^{k+2},
$$
and
$$
\lVert u - R_H(\hat{\Pi}^k_H u) \rVert_{H^1(\Omega)} \leq C h^{k+1}.
$$
Note that, for $k \geq 1$, the reconstruction is continuous and the $H^1$-norm is not a broken norm in that case.
Below, we test convergence of $R_H(\hat{\Pi}^k_H u)$ towards $u$ as the cell size decreases for the case $u(x) = \cos(4\pi x)$.

In [ ]:
test_HHO_convergence("Reconstruction", test_degrees=[1, 2, 5, 6])

The convergence rate matches the theoretically predicted value for all degrees and in both the $L^2$- and $H^1$-norm. The highest-degree approximations again suffer from the accumulation of round-off errors when the approximation is already small.

# Stabilization
The stabilization operator is the last element in the formulation of the cell problems for the HHO method. For $k=0$, we can avoid the computation of the stabilization operator (which would be one number per face) because we have an explicit formula for the HHO approximation. 
When $k \geq 1$, however, computation the stabilization matrix is inevitable to obtain a well-posed cell problem. 

We fix a cell $T$. The stabilization operator consists of two parts, $S_0$ and $S_1$, which are defined as (with the notation $\hat{u} = (u_T, u_{\partial T})$)
$$
S_0(\hat{u}) = \Pi^k_{\partial T}u_T - u_{\partial T}, \\
S_1(\hat{u}) = \Pi^k_{\partial T}{\left( R_T\hat{u} - \Pi^k_T R_T \hat{u} \right)}.
$$

We would like to point out that the first operator, $S_0$, alone is sufficient to obtain a well-posed cell problem. 
The second operator, known as the high-order contribution, serves to improve the approximation properties of the local problems by 1 order and take advantage of the fact that the reconstruction is computed in the space $\mathbb{P}_{k+1}(T)$.
Let us check these properties.

We first deactivate $S_1$ in the local problems and compute the solution to the local cell problems with a known solution $(\hat{\Pi}^k_T u, u\vert_{\partial T})$ for some function $u \in H^1(T)$. 
We solve the cell problem on the HHO space and plot the associated reconstruction, which should approximate $u$.

In [ ]:
pcell_stab_low = HHO_cell(xL,xR,k,basis_type,stabilization=0)
test_HHO_computation("Cell problem", pcell_stab_low, xx)

We see that the computed reconstruction is still exact at the two end points of the cell. 
It is also exact when the analytical solution is a 5th order polynomial, but not when it is a 6th order polynomial.

By adding the operator $S_1$ to the stabilization, we expect to also become exact for the 6th order case. This is indeed the case, as is shown below:

In [ ]:
test_HHO_computation("Cell problem", pcell, xx)

# Convergence of the HHO approximation

In [ ]:
test_HHO_convergence("Poisson solve", test_degrees=[1,2,5,6])